# Practice Lab: Long-Running & Async Agents (Lesson 11.9)

Module 11 . 8 exercises . Celery + SSE + Inngest + idempotency + circuit breakers + Temporal + events + GCP India


# Lesson 11.9: Long-Running & Async Agents

8 exercises: 2E + 3M + 3C

Exercises 1-5 run **locally** (pure Python). Ex 6-8 are design.


In [ ]:
import json, time, hashlib
import numpy as np
np.random.seed(42)


---
## Exercise 1 (Easy): Celery LLM Worker



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np; np.random.seed(42)
print("Celery LLM Worker:")
for k,v,r in [("prefetch_multiplier","1","Prevents task starvation"),("acks_late","True","Survives worker crash"),("pool","gevent","I/O-bound LLM calls")]:
    print(f"  {k}={v} -> {r}")

durs=np.random.uniform(0.5,3.0,10)
print(f"\n  10 tasks: sequential={sum(durs):.1f}s, parallel={max(durs):.1f}s, speedup={sum(durs)/max(durs):.1f}x")
print(f"  Prefetch=4: Worker grabs 4, slow LLM blocks 3 behind it")
print(f"  Prefetch=1: Each worker grabs 1, others grab remaining")
print(f"  Canvas: chain(a,b,c)=sequential | group(a,b,c)=parallel | chord(group,d)=fan-out+aggregate")


</details>


---
## Exercise 2 (Easy): SSE Streaming Endpoint



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class SSE:
    def __init__(self,total): self.total=total; self.events=[]
    def emit(self,eid,etype,data):
        self.events.append({"id":eid,"event":etype})
        return f"id: {eid}\nevent: {etype}\ndata: {json.dumps(data)}\n"
    def stream(self,start=0):
        for s in range(start,self.total):
            yield self.emit(str(s+1),"progress",{"step":s+1,"total":self.total,"pct":round((s+1)/self.total*100,1)})
        yield self.emit(str(self.total+1),"complete",{"status":"done"})

print("SSE Streaming:")
sse=SSE(5)
print("  Initial:")
for e in sse.stream(0): print(f"    {e.strip()}")
print("  Reconnect (Last-Event-ID: 3):")
sse2=SSE(5)
for e in sse2.stream(3): print(f"    {e.strip()}")
print("  SSE: unidirectional, auto-reconnect | WS: bidirectional, voice/collab")


</details>


---
## Exercise 3 (Medium): Inngest Step Memoization



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Inngest:
    def __init__(self): self.memo={}; self.tokens=0
    def step(self,name,fn,tok=1000):
        if name in self.memo: print(f"    {name}: MEMOIZED (0 tok)"); return self.memo[name]
        r=fn(); self.memo[name]=r; self.tokens+=tok; print(f"    {name}: EXECUTED ({tok} tok)"); return r
    def run(self,crash_at=None):
        steps=[("decompose",500),("search",200),("evaluate",800),("extract",1500),("synthesize",2000)]
        for i,(n,t) in enumerate(steps):
            if crash_at and i==crash_at: print(f"    {n}: CRASH!"); return False
            self.step(n,lambda:f"done-{n}",t)
        return True

print("Inngest Step Memoization:")
wf=Inngest()
print("  Run 1 (crash at step 3):"); wf.run(crash_at=3); t1=wf.tokens
print(f"  Tokens: {t1}")
print("  Run 2 (retry, 0-2 memoized):"); wf.run(); t2=wf.tokens
celery=5000*2; print(f"\n  Celery(no memo):{celery} | Inngest(memo):{t2} | Saved:{(1-t2/celery)*100:.0f}%")


</details>


---
## Exercise 4 (Medium): Idempotent Agent Actions



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib

class IdemDB:
    def __init__(self): self.recs={}; self.seen=set(); self.calls=0
    def upsert(self,key,data,idem_key):
        self.calls+=1
        if idem_key in self.seen: print(f"    SKIP: {key}"); return
        self.seen.add(idem_key); self.recs[key]=data; print(f"    UPSERT: {key}")

def mk_key(wf,act): return hashlib.sha256(f"{wf}-{act}".encode()).hexdigest()[:16]

print("Idempotent Agent Actions:")
db=IdemDB()
actions=[("send_email",{"to":"user@netsetos.com"}),("update_status",{"status":"done"}),("log",{"result":"42 findings"})]
print("  First:"); [db.upsert(n,d,mk_key("wf-123",n)) for n,d in actions]
print("  Retry:"); [db.upsert(n,d,mk_key("wf-123",n)) for n,d in actions]
print(f"\n  Calls:{db.calls} Unique:{len(db.recs)} Skipped:{db.calls-len(db.recs)}")
print(f"  Pattern: at-least-once + idempotent consumers = effectively-once")


</details>


---
## Exercise 5 (Medium): Circuit Breaker Fallback



In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import numpy as np; np.random.seed(42); import time

class CB:
    def __init__(self,name,fm=3):
        self.name=name;self.state="CLOSED";self.fm=fm;self.fc=0;self.lf=0
        self.stats={"ok":0,"fail":0,"blocked":0}
    def call(self,rate=0.9):
        if self.state=="OPEN":
            if time.time()-self.lf>0.01: self.state="HALF_OPEN"
            else: self.stats["blocked"]+=1; return None
        if np.random.random()<rate:
            self.fc=0;self.stats["ok"]+=1
            if self.state=="HALF_OPEN": self.state="CLOSED"
            return self.name
        self.fc+=1;self.stats["fail"]+=1;self.lf=time.time()
        if self.fc>=self.fm: self.state="OPEN"
        return None

print("Circuit Breaker Fallback:")
oai=CB("OpenAI"); ant=CB("Anthropic")
chain=[oai,ant]

print("  Normal (both healthy):")
for i in range(5):
    for b in chain:
        r=b.call(0.95)
        if r: print(f"    Call {i+1}: {r} [{oai.state}|{ant.state}]"); break

print("  OpenAI degraded:")
for i in range(8):
    for j,b in enumerate(chain):
        rate=0.1 if j==0 else 0.95
        r=b.call(rate)
        if r: print(f"    Call {i+6}: {r} [{oai.state}|{ant.state}]"); break
    else: print(f"    Call {i+6}: CACHE [{oai.state}|{ant.state}]")

print(f"\n  OpenAI: {oai.stats} | Anthropic: {ant.stats}")


</details>


---
## Exercise 6 (Challenge): Temporal Agent Workflow
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Temporal Agent Workflow:")
print("  5 Activities: decompose->search->evaluate->extract->synthesize")
print("  Heartbeats: activity.heartbeat() during long search")
print("  HITL: workflow.wait_condition(approved, timeout=1h)")
print("  Child: workflow.execute_child_workflow(SubResearch)")
print("  Crash recovery: event-sourced replay, zero re-execution")


</details>


---
## Exercise 7 (Challenge): Event-Driven Multi-Agent
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("Event-Driven Multi-Agent:")
print("  Fan-out: orchestrator -> Pub/Sub -> 3 agents (financial/technical/competitive)")
print("  Fan-in: all publish to results topic, synthesizer waits 3/3 (5min timeout)")
print("  Backpressure: semaphore=5 concurrent, queue>100 pause, token bucket 10/min")
print("  Event sourcing: [t1]dispatched [t2]agent-A.started [t3]llm_called [t4]completed")


</details>


---
## Exercise 8 (Challenge): Production GCP India Deploy
Design/architecture.


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
print("GCP India Production Deploy:")
print("  Architecture: Cloud Run Jobs (asia-south1) + Pub/Sub + GCS + Scheduler + DLQ")
print("  Spot: $0.0058/hr vs $0.0279/hr on-demand (79% savings)")
print("  Graceful 120s: SIGTERM -> stop accepting -> wait in-flight -> checkpoint -> exit")
print("  DLQ: alert on count>0, agent failures: context overflow, hallucination, timeout")
costs=[("Cloud Run 2vCPU",45,90),("Pub/Sub 100K/day",5,10),("GCS 50GB",1,2),("Monitoring",0,15),("LLM API",50,200)]
total_lo=sum(l for _,l,_ in costs); total_hi=sum(h for _,_,h in costs)
print(f"  Monthly: ${total_lo}-${total_hi} | DPDP: Mumbai deploy, 72hr breach, Rs 250cr max")


</details>
